<a href="https://colab.research.google.com/github/vanashri-18/CSA6101-Digital-Forensics-and-Cybercrime-Investigation/blob/main/Ransomware_Activity_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Aim**

To develop a Python program that analyzes simulated file-system events and determines whether the observed activity resembles ransomware behavior by examining rapid file modifications, mass renaming, unusual extension changes, and repeated operations across multiple directories.

**Algorithm**

Create a simulated sequence of file-system events.

Read the timestamp, event type, file path, and extension.

Count files modified within a short time interval.

Detect rapid file-renaming activity.

Identify unusual extension changes such as .docx → .locked.

Count affected directories.

Assign points to each detected ransomware indicator.

Combine the indicators to calculate a risk score.

Classify the activity as LOW, MEDIUM, HIGH, or CRITICAL.

Display the detected indicators and supporting evidence.

In [1]:
# ==============================================
# RANSOMWARE-LIKE ACTIVITY DETECTOR
# ==============================================

import pandas as pd

# ------------------------------------------------
# 1. Simulated File-System Events
# ------------------------------------------------

data = [
    ["11:00:01", "MODIFY", "C:/Finance/report1.docx"],
    ["11:00:02", "MODIFY", "C:/Finance/report2.docx"],
    ["11:00:03", "MODIFY", "C:/Finance/report3.xlsx"],
    ["11:00:04", "RENAME", "C:/Finance/report1.docx -> report1.locked"],
    ["11:00:05", "RENAME", "C:/Finance/report2.docx -> report2.locked"],
    ["11:00:06", "RENAME", "C:/Finance/report3.xlsx -> report3.locked"],
    ["11:00:07", "MODIFY", "C:/HR/employee1.docx"],
    ["11:00:08", "MODIFY", "C:/HR/employee2.pdf"],
    ["11:00:09", "RENAME", "C:/HR/employee1.docx -> employee1.locked"],
    ["11:00:10", "MODIFY", "C:/Projects/design.pptx"],
    ["11:00:11", "RENAME", "C:/Projects/design.pptx -> design.locked"],
    ["11:00:12", "MODIFY", "C:/Projects/data.xlsx"]
]

df = pd.DataFrame(
    data,
    columns=[
        "Time",
        "Event",
        "Path"
    ]
)

print("=" * 90)
print("              RANSOMWARE-LIKE ACTIVITY DETECTOR")
print("=" * 90)

# ------------------------------------------------
# 2. Detection Thresholds
# ------------------------------------------------

MODIFICATION_THRESHOLD = 5
RENAME_THRESHOLD = 3
DIRECTORY_THRESHOLD = 3

# ------------------------------------------------
# 3. Extract Directories
# ------------------------------------------------

df["Directory"] = df["Path"].apply(
    lambda x: x.split(" -> ")[0].rsplit("/", 1)[0]
)

# ------------------------------------------------
# 4. Indicator Detection
# ------------------------------------------------

indicators = []
score = 0

# -----------------------------------------------
# Indicator 1: Mass File Modification
# -----------------------------------------------

modified_count = len(
    df[df["Event"] == "MODIFY"]
)

if modified_count >= MODIFICATION_THRESHOLD:

    indicators.append(
        f"High number of file modifications: "
        f"{modified_count}"
    )

    score += 3

# -----------------------------------------------
# Indicator 2: Rapid Renaming
# -----------------------------------------------

rename_count = len(
    df[df["Event"] == "RENAME"]
)

if rename_count >= RENAME_THRESHOLD:

    indicators.append(
        f"Rapid file renaming detected: "
        f"{rename_count} files"
    )

    score += 3

# -----------------------------------------------
# Indicator 3: Unusual Extension Changes
# -----------------------------------------------

unusual_extensions = []

for _, row in df.iterrows():

    if row["Event"] == "RENAME":

        if " -> " in row["Path"]:

            old_file, new_file = row["Path"].split(
                " -> "
            )

            old_ext = "." + old_file.split(".")[-1]
            new_ext = "." + new_file.split(".")[-1]

            if new_ext in [
                ".locked",
                ".encrypted",
                ".enc",
                ".ransom"
            ]:

                unusual_extensions.append(
                    row["Path"]
                )

if unusual_extensions:

    indicators.append(
        "Unusual encrypted/locked extension changes detected"
    )

    score += 4

# -----------------------------------------------
# Indicator 4: Multiple Directories
# -----------------------------------------------

directory_count = df["Directory"].nunique()

if directory_count >= DIRECTORY_THRESHOLD:

    indicators.append(
        f"Repeated activity across "
        f"{directory_count} directories"
    )

    score += 3

# ------------------------------------------------
# 5. Risk Assessment
# ------------------------------------------------

if score >= 10:

    risk = "CRITICAL"

elif score >= 7:

    risk = "HIGH"

elif score >= 4:

    risk = "MEDIUM"

else:

    risk = "LOW"

# ------------------------------------------------
# 6. Display Event Log
# ------------------------------------------------

print("\n" + "-" * 90)
print("                    FILE-SYSTEM EVENTS")
print("-" * 90)

print(
    df[
        ["Time", "Event", "Path"]
    ].to_string(index=False)
)

# ------------------------------------------------
# 7. Display Indicators
# ------------------------------------------------

print("\n" + "=" * 90)
print("                  DETECTED INDICATORS")
print("=" * 90)

if indicators:

    for number, indicator in enumerate(
        indicators, 1
    ):

        print(
            f"{number}. {indicator}"
        )

else:

    print("No significant ransomware-like indicators detected.")

# ------------------------------------------------
# 8. Final Risk Assessment
# ------------------------------------------------

print("\n" + "=" * 90)
print("                    RISK ASSESSMENT")
print("=" * 90)

print("Risk Score :", score)
print("Risk Level :", risk)

print("\nFiles Modified :", modified_count)
print("Files Renamed  :", rename_count)
print("Directories Affected :", directory_count)

print("\nAnalysis completed.")
print("=" * 90)

              RANSOMWARE-LIKE ACTIVITY DETECTOR

------------------------------------------------------------------------------------------
                    FILE-SYSTEM EVENTS
------------------------------------------------------------------------------------------
    Time  Event                                      Path
11:00:01 MODIFY                   C:/Finance/report1.docx
11:00:02 MODIFY                   C:/Finance/report2.docx
11:00:03 MODIFY                   C:/Finance/report3.xlsx
11:00:04 RENAME C:/Finance/report1.docx -> report1.locked
11:00:05 RENAME C:/Finance/report2.docx -> report2.locked
11:00:06 RENAME C:/Finance/report3.xlsx -> report3.locked
11:00:07 MODIFY                      C:/HR/employee1.docx
11:00:08 MODIFY                       C:/HR/employee2.pdf
11:00:09 RENAME  C:/HR/employee1.docx -> employee1.locked
11:00:10 MODIFY                   C:/Projects/design.pptx
11:00:11 RENAME  C:/Projects/design.pptx -> design.locked
11:00:12 MODIFY                   

**Result**

The Python program successfully analyzed the simulated file-system activity and detected multiple ransomware-like indicators, including rapid file modifications, mass renaming, unusual .locked extensions, and activity across multiple directories. The indicators were combined into a risk score of 13, resulting in a CRITICAL assessment for the simulated dataset. This is a behavioral assessment and does not by itself prove that ransomware is present; the flagged activity should be investigated with additional forensic evidence.